In [1]:
import os
import pandas as pd
from dotenv import load_dotenv, find_dotenv
from sqlalchemy import create_engine
from sqlalchemy.engine import URL

load_dotenv(find_dotenv())   # walks up from notebooks/ to find the .env in the project root

url = URL.create(
    "postgresql+psycopg2",
    username=os.getenv("PG_USER"),
    password=os.getenv("PG_PASSWORD"),
    host=os.getenv("PG_HOST"),
    port=int(os.getenv("PG_PORT")),
    database=os.getenv("PG_DB"),
)
engine = create_engine(url)

df = pd.read_sql("""
    SELECT f.*, h.holdout_gap_days, h.holdout_is_capped, h.eval_set
    FROM features.user_features f
    JOIN features.user_holdout h USING (user_id)
""", engine)

print(df.shape)
df.info()

(206209, 24)
<class 'pandas.DataFrame'>
RangeIndex: 206209 entries, 0 to 206208
Data columns (total 24 columns):
 #   Column                Non-Null Count   Dtype  
---  ------                --------------   -----  
 0   user_id               206209 non-null  int64  
 1   n_prior_orders        206209 non-null  int64  
 2   n_gaps                206209 non-null  int64  
 3   tenure_days_lb        206209 non-null  float64
 4   mean_gap              206209 non-null  float64
 5   median_gap            206209 non-null  float64
 6   sd_gap                206209 non-null  float64
 7   mean_gap_uncapped     197412 non-null  float64
 8   n_capped_gaps         206209 non-null  int64  
 9   pct_capped_gaps       206209 non-null  float64
 10  is_censored_heavy     206209 non-null  bool   
 11  n_same_day_gaps       206209 non-null  int64  
 12  last_prior_gap        206209 non-null  float64
 13  recent_mean_gap       206209 non-null  float64
 14  gap_trend             206209 non-null  float64
 15

In [2]:
print(df.isna().sum()[lambda s: s > 0])
print("users with sd_gap == 0:", (df["sd_gap"] == 0).sum())
print("users with all gaps capped:", df["mean_gap_uncapped"].isna().sum())

mean_gap_uncapped    8797
dtype: int64
users with sd_gap == 0: 9571
users with all gaps capped: 8797
